# Les news du pétrole : le rapport EIA complet, décomposé par type de choc
**Pourquoi pas des articles ?** Aucune source gratuite ne fournit des articles pétrole datés à la minute sur 10 ans (GDELT : 12 mois sans compte ; CrudeOilNews : 24 articles datés).

**L'idée :** le rapport EIA du mercredi (10h30, heure de New York) ne contient pas qu'un chiffre. Il en contient huit, et chacun correspond à un type de choc du README :

| Type de choc | Composantes | Haussier si… |
|---|---|---|
| **Stockage** | stocks de brut, stocks à Cushing (point de livraison du WTI) | les stocks baissent plus que d'habitude |
| **Offre** | production US, importations | elles baissent |
| **Demande** | consommation (product supplied), utilisation des raffineries, stocks d'essence et de distillats | la consommation monte, les stocks de produits baissent |

Pour chaque composante : **surprise = variation de la semaine − variation habituelle** (moyenne de la même semaine sur les 5 ans précédents), mise en écarts-types et signée pour que **+ = haussier**. On obtient le vecteur de news $N_t$ du README, mesuré au lieu d'être extrait d'un texte.

**Tests fixés à l'avance (avant de regarder les résultats) :**
1. Quelles composantes font bouger le WTI dans les 5 minutes ?
2. Stratégie de 10h35 à 10h45 : le vecteur de news complet, puis les régimes (courbe, HMM), font-ils mieux que le brut seul ? (2016-2024)
3. Test final sur des données jamais utilisées (avril 2024 → aujourd'hui), lancé une seule fois.

In [1]:
# À lancer UNE SEULE FOIS : les 8 composantes du rapport EIA hebdomadaire -> data/eia_components.csv
import os, warnings
warnings.filterwarnings("ignore")                # cache les avertissements "rank-deficient" (fenêtres sans marché tendu)
os.environ["LOKY_MAX_CPU_COUNT"] = "4"           # cache le warning joblib sous Windows
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

CODES = {"brut": "WCESTUS1", "cushing": "W_EPC0_SAX_YCUOK_MBBL", "essence": "WGTSTUS1", "distillats": "WDISTUS1",
         "production": "WCRFPUS2", "imports": "WCRIMUS2", "raffineries": "WPULEUS3", "conso": "WRPUPUS2"}
if not os.path.exists("data/eia_components.csv"):
    comp = {}
    for k, c in CODES.items():
        try:
            comp[k] = pd.read_excel(f"https://www.eia.gov/dnav/pet/hist_xls/{c}w.xls", sheet_name="Data 1", skiprows=2,
                                    index_col=0, storage_options={"User-Agent": "Mozilla/5.0"}).squeeze("columns")
        except Exception as e:
            print("échec :", k, c, e)
    pd.DataFrame(comp).to_csv("data/eia_components.csv")

## Construction des scores de news
Crée `data/news_eia.csv` : une ligne par rapport, avec les 8 composantes et les 3 scores (`score_offre`, `score_demande`, `score_stockage`).

In [2]:
comp = pd.read_csv("data/eia_components.csv", index_col=0, parse_dates=True)
SIGNE = {"brut": -1, "cushing": -1, "essence": -1, "distillats": -1, "production": -1, "imports": -1, "raffineries": 1, "conso": 1}
TYPES = {"stockage": ["brut", "cushing"], "offre": ["production", "imports"], "demande": ["conso", "raffineries", "essence", "distillats"]}

d = comp.diff()
seas = d.groupby(d.index.isocalendar().week).transform(lambda s: s.shift(1).rolling(5).mean())
surp = d - seas                                          # surprise = variation - variation habituelle de cette semaine
news = surp / surp.expanding(104).std().shift(1)         # en écarts-types, calculés sur le passé uniquement
news = news * pd.Series(SIGNE)                           # convention : + = haussier pour le prix
for t, cols in TYPES.items():
    news["score_" + t] = news[cols].mean(axis=1)         # un score par type de choc
news.index = news.index + pd.Timedelta(days=5)           # date de publication (mercredi 10h30 ET)
news.loc["2013":].dropna(how="all").to_csv("data/news_eia.csv")

## Ouvrir le fichier propre

In [3]:
news = pd.read_csv("data/news_eia.csv", index_col=0, parse_dates=True)
news.round(2).tail()

,brut,cushing,essence,distillats,production,imports,raffineries,conso,score_stockage,score_offre,score_demande
Date,,,,,,,,,,,
2026-09-23,-0.77,-1.53,0.38,-0.35,0.60,2.12,-0.78,-0.69,-1.15,1.36,-0.36
2026-09-30,-0.01,-0.03,1.53,0.49,0.30,-0.19,-0.26,0.94,-0.02,0.06,0.67
2026-10-07,2.25,-0.59,-0.91,-1.05,0.55,-1.87,0.69,-1.33,0.83,-0.66,-0.65
2026-10-14,1.63,-1.70,-2.29,-0.22,2.17,0.49,1.18,0.42,-0.04,1.33,-0.22
2026-10-21,1.22,-1.24,0.83,-1.17,2.76,1.04,-0.45,0.78,-0.01,1.90,-0.00


## Test 1 : quelles news font bouger le prix en 5 minutes ?
Rendement 10h29 → 10h34 sur les 8 composantes. Avec la convention + = haussier, les coefficients devraient être **positifs**. Le test F dit si les 7 autres composantes apportent quelque chose en plus des stocks de brut.

Nécessite `data/eia_intraday.csv` (et `data/eia_intraday_recent.csv` s'il existe, créé par `test_2`).

In [4]:
df = pd.read_csv("data/dataset.csv", index_col=0, parse_dates=True)
intra = pd.concat([pd.read_csv(f, index_col=0, parse_dates=True)
                   for f in ["data/eia_intraday.csv", "data/eia_intraday_recent.csv"] if os.path.exists(f)])
ev = intra.join(news, how="inner").join((df["slope"].shift(1) > 0).astype(int).rename("B"))   # B connu jusqu'en avril 2024
ev["y"] = ev["r15"] - ev["r5"]                                                                 # rendement tradable 10h35 -> 10h45

COMPOS = list(SIGNE)
m = smf.ols("r5 ~ " + " + ".join(COMPOS), data=ev).fit(cov_type="HAC", cov_kwds={"maxlags": 4})
print(m.summary().tables[1])
print("\nLes 7 autres composantes ajoutent-elles quelque chose au brut ?\n", m.f_test(", ".join(f"{c} = 0" for c in COMPOS[1:])))

                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -0.0321      0.023     -1.392      0.164      -0.077       0.013
brut            0.1232      0.026      4.828      0.000       0.073       0.173
cushing         0.0076      0.026      0.288      0.773      -0.044       0.059
essence         0.1241      0.032      3.882      0.000       0.061       0.187
distillats      0.0630      0.029      2.163      0.031       0.006       0.120
production      0.0452      0.020      2.250      0.024       0.006       0.085
imports         0.0918      0.029      3.196      0.001       0.035       0.148
raffineries     0.0020      0.027      0.077      0.939      -0.050       0.054
conso           0.0177      0.018      0.973      0.330      -0.018       0.053

Les 7 autres composantes ajoutent-elles quelque chose au brut ?
 <F test: F=7.5373383504701215, p=1.125345420212448e-08

## Régimes de marché : probabilités du HMM (sans information future)
Même HMM que dans `test_2` : estimé sur 2007-2018, probabilité de la veille de chaque rapport.

In [5]:
from hmmlearn.hmm import GaussianHMM
from scipy.stats import multivariate_normal
from sklearn.preprocessing import StandardScaler

feat = pd.read_csv("data/wti_hmm_features.csv", index_col=0, parse_dates=True).sort_index()
sc = StandardScaler().fit(feat[:"2018"])
X = sc.transform(feat)
hmm = GaussianHMM(3, covariance_type="full", n_iter=500, random_state=42).fit(X[feat.index <= "2018-12-31"])
ll = np.column_stack([multivariate_normal(hmm.means_[k], hmm.covars_[k]).logpdf(X) for k in range(3)])
a, p = np.zeros_like(ll), hmm.startprob_
for t in range(len(X)):                                  # filtre de Hamilton : P(régime | infos jusqu'à t)
    w = p * np.exp(ll[t] - ll[t].max())
    a[t] = w / w.sum()
    p = a[t] @ hmm.transmat_
order = np.argsort(sc.inverse_transform(hmm.means_)[:, 1])          # 0 calme, 1 normal, 2 crise
P = pd.DataFrame(a[:, order], index=feat.index, columns=["P0", "P1", "P2"]).shift(1)   # veille de l'événement
ev = ev.drop(columns=["P0", "P1", "P2"], errors="ignore").join(P.reindex(ev.index, method="ffill"))

## Test 2 : stratégie 10h35 → 10h45 avec news et régimes (2016-2024)
Chaque année, le modèle est estimé sur les 3 années précédentes ; achat si la dérive prédite est positive, vente sinon ; coût 0,04 %.
`t-stat écart vs brut seul` > 2 → significativement mieux que le modèle de base.

In [6]:
COST, S3 = 0.04, "(score_offre + score_demande + score_stockage)"

def backtest(f, data, annees, debut=None):
    pnl = []
    for yr in annees:                                    # chaque année : modèle estimé sur les 3 années d'avant
        tr, te = data.loc[str(yr - 3):str(yr - 1)], data.loc[str(yr)]
        te = te.loc[debut:] if debut else te
        pnl.append(np.sign(smf.ols(f, data=tr).fit().predict(te)) * te["y"] - COST)
    return pd.concat(pnl)

def perf(p, ref=None):
    ans = (p.index[-1] - p.index[0]).days / 365.25
    r = {"trades": len(p), "Sharpe net": p.mean() / p.std() * np.sqrt(len(p) / ans), "t-stat": p.mean() / p.std() * np.sqrt(len(p))}
    if ref is not None:
        diff = p - ref
        r["t-stat écart vs brut seul"] = diff.mean() / diff.std() * np.sqrt(len(diff))
    return r

dev = ev.loc[:"2024-04-05"].dropna(subset=["y", "B", "P1", "score_offre", "score_demande", "score_stockage"])
modeles = {"brut seul": "y ~ brut", "3 scores de news": "y ~ " + S3,
           "3 scores × courbe (B)": f"y ~ {S3} * B", "3 scores × HMM": f"y ~ {S3} * (P1 + P2)"}
pnl = {k: backtest(f, dev, range(2016, 2025)) for k, f in modeles.items()}
pd.DataFrame({k: perf(p, None if k == "brut seul" else pnl["brut seul"]) for k, p in pnl.items()}).T.round(2)

,trades,Sharpe net,t-stat,t-stat écart vs brut seul
brut seul,365.0,0.60,1.72,NaN
3 scores de news,365.0,-0.41,-1.17,-2.87
3 scores × courbe (B),365.0,-0.36,-1.03,-2.69
3 scores × HMM,365.0,0.07,0.20,-1.79


## Test 3 : test final, avril 2024 → aujourd'hui
À lancer **une seule fois**, sans rien modifier ensuite. Les régimes ne sont pas disponibles après avril 2024 (fin des prix M1/M2 de l'EIA) : on compare seulement le brut seul et les 3 scores de news.

In [7]:
ho = ev.dropna(subset=["y", "score_offre", "score_demande", "score_stockage"])
res = {k: perf(backtest(modeles[k], ho, range(2024, pd.Timestamp.today().year + 1), debut="2024-04-06"))
       for k in ["brut seul", "3 scores de news"]}
pd.DataFrame(res).T.round(2)

,trades,Sharpe net,t-stat
brut seul,115.0,-0.92,-1.47
3 scores de news,115.0,-0.86,-1.37
